In [0]:
dbutils.widgets.text("catalog", "vp_rm20040829")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

df = spark.read.table(f"{catalog}.bronze.sessions")

# --- standardization ---
df = (
    df.withColumn("session_id", upper(trim(col("session_id"))).cast("string"))
    .withColumn("charger_id", upper(trim(col("charger_id"))).cast("string"))
    .withColumn("member_id", upper(trim(col("member_id"))).cast("string"))
    .withColumn("started_at", coalesce(
        try_to_timestamp(col("started_at"), lit("yyyy-MM-dd HH:mm:ss")),
        try_to_timestamp(col("started_at"), lit("yyyy-MM-dd'T'HH:mm:ss"))
    ))
    .withColumn("ended_at", coalesce(
        try_to_timestamp(col("ended_at"), lit("yyyy-MM-dd HH:mm:ss")),
        try_to_timestamp(col("ended_at"), lit("yyyy-MM-dd'T'HH:mm:ss"))
    ))
    .withColumn("status", upper(trim(col("status"))).cast("string"))
    .withColumn("updated_at", coalesce(
        try_to_timestamp(col("updated_at"), lit("yyyy-MM-dd HH:mm:ss")),
        try_to_timestamp(col("updated_at"), lit("yyyy-MM-dd'T'HH:mm:ss"))
    ))
    .withColumn("session_date", to_date(col("started_at")))
    .withColumn("energy_kwh",
        when(get_json_object(col("meter"), "$.start_kwh").isNotNull() & get_json_object(col("meter"), "$.end_kwh").isNotNull(),
             get_json_object(col("meter"), "$.end_kwh").cast("decimal(10,3)") - get_json_object(col("meter"), "$.start_kwh").cast("decimal(10,3)"))
        .otherwise(lit(None))
        .cast("decimal(10,3)")
    )
    .withColumn("amount_paid", get_json_object(col("payment"), "$.amount_inr").cast("decimal(10,2)"))
    .withColumn("idle_minutes", coalesce(col("idle_minutes"), lit(0)).cast("int"))
    .withColumn("time_band",
        when(col("started_at").isNotNull() & (hour(col("started_at")).between(18, 21)), "PEAK")
        .otherwise("OFF_PEAK")
    )
    .withColumn("duration_minutes",
        when(
            col("ended_at").isNotNull() & col("started_at").isNotNull() & (col("ended_at") > col("started_at")),
            floor(unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at")) / 60).cast("int")
        )
        .otherwise(lit(None).cast("int"))
    )
)

w = Window.partitionBy("session_id").orderBy(
    col("updated_at").desc_nulls_last()
)

latest = (
    df
    .filter(col("session_id").isNotNull())
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

# --- lookup sets ---
valid_charger_ids = [
    row[0]
    for row in spark.table(f"{catalog}.silver.chargers")
    .select("charger_id")
    .distinct()
    .collect()
]

valid_member_ids = [
    row[0]
    for row in spark.table(f"{catalog}.silver.members")
    .select("member_id")
    .distinct()
    .collect()
]

# --- validation ---
checked = latest.withColumn(
    "validation_error",
    when(
        col("charger_id").isNull() | ~col("charger_id").isin(valid_charger_ids),
        "UNKNOWN_CHARGER"
    )
    .when(
        col("member_id").isNotNull() & ~col("member_id").isin(valid_member_ids),
        "UNKNOWN_MEMBER"
    )
    .when(
        col("started_at").isNull(),
        "INVALID_TIMESTAMP"
    )
    .when(
        col("status").isin("COMPLETED", "FAILED") &
        (col("ended_at").isNull() | (col("ended_at") <= col("started_at"))),
        "INVALID_DURATION"
    )
    .when(
        (col("status") == "COMPLETED") &
        (col("energy_kwh").isNull() | (col("energy_kwh") < 0)),
        "INVALID_METER"
    )
)

valid_sessions = checked.filter(col("validation_error").isNull())
quarantine_sessions = checked.filter(col("validation_error").isNotNull())

# --- silver.sessions ---
silver_sessions = valid_sessions.select(
    "session_id",
    "charger_id",
    "member_id",
    "started_at",
    "ended_at",
    "session_date",
    "status",
    "time_band",
    "energy_kwh",
    "duration_minutes",
    "idle_minutes",
    "amount_paid",
    "updated_at"
)

silver_sessions.write \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.sessions")

faults_df = valid_sessions.select(
    "session_id",
    posexplode(from_json(col("faults"), "array<string>")).alias("pos", "fault")
)

silver_session_faults = (
    faults_df
    .withColumn("fault_seq", (col("pos") + 1).cast("int"))
    .withColumn("fault_code", upper(trim(col("fault"))).cast("string"))
    .select("session_id", "fault_seq", "fault_code")
)

silver_session_faults.write \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.session_faults")

quarantine_df = quarantine_sessions.select(
    col("session_id"),
    col("validation_error").alias("reject_reason"),
    col("updated_at"),
    to_json(struct("*")).alias("raw_record"),
    current_timestamp().alias("quarantined_at")
)

quarantine_df.write \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.quarantine_sessions")

In [0]:
%sql
desc history vp_rm20040829.silver.sessions

In [0]:
%sql
select count(*) from vp_rm20040829.silver.sessions

In [0]:
%sql
select count(*) from vp_rm20040829.silver.session_faults

In [0]:
%sql
select count(*) from vp_rm20040829.silver.quarantine_sessions

In [0]:
%sql
select * from vp_rm20040829.silver.quarantine_sessions

In [0]:
spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.sessions IS 'This table contains transformed sessions';
""");

spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.session_faults IS 'This table contains transformed session_faults';
""");

spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.quarantine_sessions IS 'This table contains quarantined sessions';
""")